# 2층 신경망 역전파 유도 (입력2-은닉2-출력1)

- 활성화 함수: sigmoid $\sigma(z)=1/(1+e^{-z})$
- 손실 함수: BCE $L=-(y\log\hat y + (1-y)\log(1-\hat y))$
- 고정 예제: $x=[1,0], W_1=[[0.1,0.2],[0.3,0.4]], b_1=[0,0], W_2=[0.5,0.6], b_2=0, y_{true}=1$


## 순전파 수식 및 shape

- $x$: (2,)
- $W_1$: (2,2), $b_1$: (2,)
- $z_1 = W_1x + b_1$: (2,)
- $a_1 = \sigma(z_1)$: (2,)
- $W_2$: (2,), $b_2$: scalar
- $z_2 = W_2\cdot a_1 + b_2$: scalar
- $\hat y = \sigma(z_2)$: scalar


## 역전파 유도 (연쇄 법칙)

1. $\frac{\partial L}{\partial \hat y}= -\left(\frac{y}{\hat y} - \frac{1-y}{1-\hat y}\right)$ (scalar)
2. BCE+Sigmoid 결합: $\frac{\partial L}{\partial z_2}=\hat y-y$ (scalar)
3. $\frac{\partial L}{\partial W_2}=\frac{\partial L}{\partial z_2} a_1$ (2,)
4. $\frac{\partial L}{\partial a_1}=\frac{\partial L}{\partial z_2} W_2$ (2,)
5. $\frac{\partial L}{\partial z_1}=\frac{\partial L}{\partial a_1} \odot a_1(1-a_1)$ (2,)
6. $\frac{\partial L}{\partial W_1}=\frac{\partial L}{\partial z_1}\otimes x$ (2,2)


In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# 고정 예제
x = np.array([1.0, 0.0])
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.array([0.0, 0.0])
W2 = np.array([0.5, 0.6])
b2 = 0.0
y_true = 1.0

# 순전파
z1 = W1 @ x + b1
a1 = sigmoid(z1)
z2 = W2 @ a1 + b2
y_pred = sigmoid(z2)

print('z1:', z1, 'shape=', z1.shape)
print('a1:', a1, 'shape=', a1.shape)
print('z2:', z2, 'shape=scalar')
print('y_pred:', y_pred, 'shape=scalar')

# 역전파
dL_dy = -(y_true / y_pred) + ((1-y_true) / (1-y_pred))
dL_dz2 = y_pred - y_true
dL_dW2 = dL_dz2 * a1
dL_da1 = dL_dz2 * W2
dL_dz1 = dL_da1 * a1 * (1-a1)
dL_dW1 = np.outer(dL_dz1, x)

print('dL/dy_pred:', dL_dy)
print('dL/dz2:', dL_dz2)
print('dL/dW2:', dL_dW2)
print('dL/da1:', dL_da1)
print('dL/dz1:', dL_dz1)
print('dL/dW1:
', dL_dW1)

# 손계산 기대값(소수점 4자리 기준)
manual = {
    'z1': np.array([0.1, 0.3]),
    'a1': np.array([0.5250, 0.5744]),
    'z2': 0.6071,
    'y_pred': 0.6473,
    'dL_dz2': -0.3527,
    'dL_dW2': np.array([-0.1852, -0.2026]),
    'dL_dz1': np.array([-0.0440, -0.0513]),
}

checks = [
    np.allclose(np.round(z1,4), np.round(manual['z1'],4)),
    np.allclose(np.round(a1,4), np.round(manual['a1'],4)),
    np.isclose(round(float(z2),4), round(manual['z2'],4)),
    np.isclose(round(float(y_pred),4), round(manual['y_pred'],4)),
    np.isclose(round(float(dL_dz2),4), round(manual['dL_dz2'],4)),
    np.allclose(np.round(dL_dW2,4), np.round(manual['dL_dW2'],4)),
    np.allclose(np.round(dL_dz1,4), np.round(manual['dL_dz1'],4)),
]
print('소수점 4자리 일치 여부:', all(checks))
